In [1]:
import pandas as pd
import re
from sqlalchemy import create_engine
import urllib

In [2]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=181.57.189.150,1433;"
    "DATABASE=Ventas_Comerssia;"
    "UID=sa;"
    "PWD=P@ssw0rd*;"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [3]:
query = """
SELECT DISTINCT c.CLICodigo, 
    c.CLINombres, 
    c.CLIApellidos, 
    c.CLIEmailPrincipal, 
    c.CLICelular,
    c.CLITipoDocumento
FROM Ventas_Comerssia.dbo.Clientes_Comerssia c
INNER JOIN Ventas_Comerssia.dbo.Ventas_Comerssia v ON c.CLICodigo = v.Cliente
WHERE Fecha >= '2025-08-11'
     AND Fecha <= '2025-08-12'
"""

# Ejecutar y cargar en DataFrame
df = pd.read_sql(query, engine)


# # Obtener los CLICodigos ya existentes en la tabla SQL
clientes_existentes = pd.read_sql("SELECT CLICodigo FROM Contacto_Clientes", con=engine)

In [4]:
#Limpiar Celular
def limpiar_celular(cel):
    if not cel:
        return ""
    cel = str(cel).strip().replace(" ", "").replace("-", "").replace("(", "").replace(")", "")
    if cel.startswith("+57"):
        cel = cel[3:]
    elif cel.startswith("57") and len(cel) > 10:
        cel = cel[2:]
    return cel

df['CLICelular'] = df['CLICelular'].apply(limpiar_celular)

#Validar celular
def es_celular_valido(cel):
    return bool(re.fullmatch(r"3\d{9}", str(cel)))

df['CelularValido'] = df['CLICelular'].apply(es_celular_valido)

#Limpiar y validar email
df['CLIEmailPrincipal'] = df['CLIEmailPrincipal'].apply(lambda x: str(x).strip().lower())

def es_email_valido(email):
    email = str(email).strip().lower()

    # Si está vacío
    if not email:
        return False
    
    # Palabras prohibidas
    palabras_no_permitidas = ["negad", "pendi"]
    if any(palabra in email for palabra in palabras_no_permitidas):
        return False
    
    # Validación de formato
    return bool(re.fullmatch(r'^[\w\.-]+@[\w\.-]+\.\w+$', email))

df['EmailValido'] = df['CLIEmailPrincipal'].apply(es_email_valido)

In [5]:
def limpiar_codigo(codigo):
    codigo = str(codigo).strip()     
    codigo = re.sub(r'^C\s*', '', codigo, flags=re.IGNORECASE)         # quita espacios al inicio y al final
    codigo = codigo.replace(' ', '')          # elimina cualquier espacio restante en el medio
    return codigo

df.insert(1, 'Documento', df['CLICodigo'].apply(limpiar_codigo))

df

,CLICodigo,Documento,CLINombres,CLIApellidos,CLIEmailPrincipal,CLICelular,CLITipoDocumento,CelularValido,EmailValido
0,C10543893,10543893,JOSE,FERNANDEZ,josedarfer@gmail.com,3142323221,CC,True,True
1,C1128266864,1128266864,LUISA,ARANGO,luisaarango@gmail.com,3183677703,CC,True,True
2,C1128416178,1128416178,LORENA,PALACIO,lorenapalaciosanchez@gmail.com,3154159381,CC,True,True
3,C1140837756,1140837756,GISELLA,ABDALA,gabdala01@gmail.com,3043887019,CC,True,True
4,C19484029,19484029,DIEGO,QUINTERO MUNERA,negado@provenzal.net,3153575075,CC,True,False
...,...,...,...,...,...,...,...,...,...
175,C66901005,66901005,YELIKZA,VALDERRUTEN VANDER HUCK,yelikza.valderruten@gmail.com,3154167350,CC,True,True
176,C66979979,66979979,ISABELLA,LOPEZ VERA,isablv@yahoo.com,3212084909,CC,True,True
177,C860024394,860024394,ORDEN CARMELITAS,DESCALZOS,economoprovincial@duruelo.com.co,3017849779,NI,True,True
178,C900840809,900840809,E2M,SAS,facturacion@e2msas.com,3004966644,NI,True,True


In [6]:
# Detectar duplicados
cel_duplicados = df['CLICelular'].duplicated(keep=False)
email_duplicados = df['CLIEmailPrincipal'].duplicated(keep=False)

# Marcar celulares duplicados como no válidos
df.loc[cel_duplicados, 'CelularValido'] = False

# Marcar correos duplicados como no válidos
df.loc[email_duplicados, 'EmailValido'] = False

In [7]:
#Agregar columna contacto
def determinar_contacto(row):
    if row['CelularValido'] and row['EmailValido']:
        return "Cel + Email"
    elif row['CelularValido']:
        return "Cel"
    elif row['EmailValido']:
        return "Email"
    else:
        return "Falso"

df['Contacto'] = df.apply(determinar_contacto, axis=1)

In [8]:
# Filtrar para dejar solo nuevos clientes
df_nuevos = df[~df["CLICodigo"].isin(clientes_existentes["CLICodigo"])]

In [9]:
# Insertar solo los nuevos en SQL
if not df.empty:
    df_nuevos.to_sql("Contacto_Clientes", con=engine, if_exists="append", index=False)
    print(f"{len(df_nuevos)} nuevos clientes insertados.")
else:
    print("No hay nuevos clientes por insertar.")

36 nuevos clientes insertados.


In [10]:
# # Exportar todos los registros con validaciones incluidas
# df.to_excel("clientes_con_validacion.xlsx", index=False)

In [11]:
# df.to_sql("Contacto_Clientes", engine, if_exists="replace", index=False)